# F7-kernels-convex-optimization — Practice p05

**Type:** constrained coding · **Difficulty:** intro · **Concepts:** positive-semidefinite-matrices

Implement `psd_report(A)`.

Input contract: `A` is a finite real-numeric NumPy array of shape `(n, n)` with `n >= 1`. Reject any other input with `ValueError`; do not mutate it.

Use `ATOL = 1e-10`, `RTOL = 0.0`. Return a dictionary with exactly these keys:

- `"symmetric"`: a plain `bool`, using `np.allclose(A, A.T, atol=ATOL, rtol=RTOL)`;
- `"min_eigenvalue"`: `None` when the matrix is not symmetric at that tolerance, otherwise a finite plain `float` equal to the smallest eigenvalue of `(A + A.T) / 2`;
- `"psd"`: a plain `bool`, true exactly when the matrix is symmetric and its reported minimum eigenvalue is at least `-ATOL`.

The symmetric average prevents a tiny accepted asymmetry from being handed directly to a symmetric eigensolver. The tolerance is part of the computational report, not a change to the exact mathematical definition.

In [ ]:
import numpy as np

ATOL = 1e-10
RTOL = 0.0

def psd_report(A):
    """Report symmetry, the least symmetric-part eigenvalue, and PSD status."""
    if (
        not isinstance(A, np.ndarray)
        or A.ndim != 2
        or A.shape[0] < 1
        or A.shape[0] != A.shape[1]
        or not np.issubdtype(A.dtype, np.number)
        or np.iscomplexobj(A)
        or not np.isfinite(A).all()
    ):
        raise ValueError("A must be a finite real-numeric nonempty square array")

    symmetric = bool(np.allclose(A, A.T, atol=ATOL, rtol=RTOL))
    if not symmetric:
        return {"symmetric": False, "min_eigenvalue": None, "psd": False}

    A_float = A.astype(float, copy=False)
    symmetric_part = (A_float + A_float.T) / 2.0
    min_eigenvalue = float(np.linalg.eigvalsh(symmetric_part)[0])
    return {
        "symmetric": True,
        "min_eigenvalue": min_eigenvalue,
        "psd": bool(min_eigenvalue >= -ATOL),
    }

## Immutable contract check — do not edit

The public fixtures cover positive definite, singular PSD, indefinite, nonsymmetric, and tolerance-scale cases, plus invalid input and non-mutation checks.

In [ ]:
_fixtures_p05 = (
    np.array([[2.0]]),
    np.array([[3.0, 0.0], [0.0, 1.0]]),
    np.array([[2.0, -2.0], [-2.0, 2.0]]),
    np.array([[1.0, 2.0], [2.0, 1.0]]),
    np.array([[1.0, 2.0 + 0.5 * ATOL], [2.0, 5.0]]),
    np.array([[0.0, 1.0], [0.0, 0.0]]),
)
for _A_p05 in _fixtures_p05:
    _before_p05 = _A_p05.copy()
    _report_p05 = psd_report(_A_p05)
    assert np.array_equal(_A_p05, _before_p05)
    assert type(_report_p05) is dict
    assert set(_report_p05) == {"symmetric", "min_eigenvalue", "psd"}
    assert type(_report_p05["symmetric"]) is bool
    assert type(_report_p05["psd"]) is bool
    _sym_p05 = bool(np.allclose(_A_p05, _A_p05.T, atol=ATOL, rtol=RTOL))
    assert _report_p05["symmetric"] is _sym_p05
    if not _sym_p05:
        assert _report_p05["min_eigenvalue"] is None
        assert _report_p05["psd"] is False
    else:
        _min_p05 = float(np.linalg.eigvalsh((_A_p05 + _A_p05.T) / 2.0)[0])
        assert type(_report_p05["min_eigenvalue"]) is float
        assert np.isfinite(_report_p05["min_eigenvalue"])
        assert np.isclose(_report_p05["min_eigenvalue"], _min_p05, atol=ATOL, rtol=RTOL)
        assert _report_p05["psd"] is bool(_min_p05 >= -ATOL)

_invalid_p05 = (
    [[1.0]],
    np.array([1.0, 2.0]),
    np.empty((0, 0)),
    np.ones((2, 3)),
    np.array([[1.0, np.nan], [0.0, 1.0]]),
    np.array([[1.0 + 0.0j]]),
    np.array([["x"]]),
)
for _bad_p05 in _invalid_p05:
    try:
        psd_report(_bad_p05)
    except ValueError:
        pass
    else:
        raise AssertionError("invalid A must raise ValueError")

### Solution reasoning

The input checks are completed before any linear algebra, so invalid shapes, nonnumeric dtypes, complex values, and nonfinite entries cannot reach the eigensolver. Symmetry is tested with the exact declared tolerances. Only an accepted matrix is averaged with its transpose; the symmetric eigensolver then returns ordered eigenvalues, making index zero the required minimum. The PSD flag applies the specified numerical boundary directly.

### Answer check

In [ ]:
_answer_singular_p05 = psd_report(np.array([[2.0, -2.0], [-2.0, 2.0]]))
assert _answer_singular_p05["symmetric"] is True
assert np.isclose(_answer_singular_p05["min_eigenvalue"], 0.0, atol=ATOL, rtol=RTOL)
assert _answer_singular_p05["psd"] is True
assert psd_report(np.array([[1.0, 2.0], [2.0, 1.0]]))["psd"] is False